# Multilingual Sentiment Analysis System (Updated)

This Jupyter Notebook walks step-by-step through building a production-quality 3-class sentiment classifier (positive, neutral, negative) using fine-tuned `xlm-roberta-base` on English and Spanish Amazon reviews from the `mteb/amazon_reviews_multi` dataset, and explainability using SHAP.

### Step 1: Install & Import Dependencies

In [1]:
import os
import re
import json
import string
from typing import Dict, List, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import shap

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check for hardware acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'torch'

### Step 2: Define Text Cleaning and Subsampling Helpers
We need functions to strip HTML tags, normalize whitespaces, and stratify our subsamples to balance representations.

In [2]:
def clean_text(text: str, lowercase: bool = False) -> str:
    """Clean raw review text."""
    if not isinstance(text, str):
        return ""
    # Strip HTML tags
    text = re.sub(r"<[^>]*>", "", text)
    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text).strip()
    if lowercase:
        text = text.lower()
    return text

def stratify_subsample(df: pd.DataFrame, target_col: str, total_samples: int) -> pd.DataFrame:
    """Subsample a dataframe stratifiably based on target label column."""
    counts = df[target_col].value_counts()
    ratios = counts / counts.sum()
    samples_per_class = (ratios * total_samples).round().astype(int)

    diff = total_samples - samples_per_class.sum()
    if diff != 0:
        samples_per_class.iloc[0] += diff

    sampled_dfs = []
    for cls, n_samples in samples_per_class.items():
        cls_df = df[df[target_col] == cls]
        n_to_sample = min(len(cls_df), n_samples)
        sampled_dfs.append(cls_df.sample(n=n_to_sample, random_state=42))

    return pd.concat(sampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

### Step 3: Load Amazon Multilingual Dataset from MTEB
We load reviews using configuration `"all_languages"` from `mteb/amazon_reviews_multi`. We will filter for English (`"en"`) and Spanish (`"es"`). We will map rating labels: 4 $\rightarrow$ positive (2), 2 $\rightarrow$ neutral (1), 0-1 $\rightarrow$ negative (0), drop 3.

In [3]:
# Set small sample size for local CPU runs (e.g., 2000 per language), or 50000 for training
SAMPLE_SIZE_PER_LANG = 2000  # Default to 2000 for quick notebook execution. Scale to 50000 for final training.

print("Loading mteb/amazon_reviews_multi dataset for all languages...")
dataset = load_dataset("mteb/amazon_reviews_multi", "all_languages", split="train")
full_df = pd.DataFrame(dataset)

languages = ["en", "es"]
dfs = []

for lang in languages:
    print(f"Filtering and processing {lang} subset...")
    df = full_df[full_df["language"] == lang].copy()
    
    # Remove label 3 (representing 4 star ratings)
    df = df[df["label"] != 3].copy()
    
    # Map stars (0-4) to class labels (2: Positive, 1: Neutral, 0: Negative)
    def map_label(label):
        if label == 4: return 2
        elif label == 2: return 1
        else: return 0
        
    df["class_label"] = df["label"].apply(map_label)
    df["cleaned_review"] = df["text"].apply(clean_text)
    df = df[df["cleaned_review"].str.strip() != ""]
    
    # Subsample
    df_sampled = stratify_subsample(df, "class_label", SAMPLE_SIZE_PER_LANG)
    dfs.append(df_sampled)

combined_df = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)
combined_df = combined_df.rename(columns={"class_label": "label"})

print(f"Combined dataset shape: {combined_df.shape}")
print("Class distribution before sampler balancing:")
print(combined_df["label"].value_counts().sort_index())

Loading mteb/amazon_reviews_multi dataset for all languages...


NameError: name 'load_dataset' is not defined

### Step 4: Split Dataset & Setup Tokenizer
We split the data into train (80%), validation (10%), and test (10%) splits. Then we load the `xlm-roberta-base` tokenizer.

In [4]:
# Split datasets
train_size = int(len(combined_df) * 0.8)
val_size = int(len(combined_df) * 0.1)

train_df = combined_df.iloc[:train_size].reset_index(drop=True)
val_df = combined_df.iloc[train_size:train_size + val_size].reset_index(drop=True)
test_df = combined_df.iloc[train_size + val_size:].reset_index(drop=True)

print(f"Dataset splits: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# Tokenizer setup
tokenizer_name = "xlm-roberta-base"
print(f"Loading tokenizer: {tokenizer_name}...")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Ensure model artifacts output dir exists
os.makedirs("model/artifacts/", exist_ok=True)
tokenizer.save_pretrained("model/artifacts/")

label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
with open("model/artifacts/label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=4)

NameError: name 'combined_df' is not defined

### Step 5: Tokenization & PyTorch Dataset Creation
We encode our text datasets into PyTorch-friendly representations.

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

MAX_LEN = 128
print("Encoding texts...")
train_encodings = tokenizer(train_df["cleaned_review"].tolist(), truncation=True, padding="max_length", max_length=MAX_LEN)
val_encodings = tokenizer(val_df["cleaned_review"].tolist(), truncation=True, padding="max_length", max_length=MAX_LEN)
test_encodings = tokenizer(test_df["cleaned_review"].tolist(), truncation=True, padding="max_length", max_length=MAX_LEN)

train_dataset = SentimentDataset(train_encodings, train_df["label"].tolist())
val_dataset = SentimentDataset(val_encodings, val_df["label"].tolist())
test_dataset = SentimentDataset(test_encodings, test_df["label"].tolist())

### Step 6: Create Dataloaders with Balanced Sampler
Using a `WeightedRandomSampler` guarantees each batch has an even representation of negative, neutral, and positive reviews during training.

In [ ]:
BATCH_SIZE = 16  # Small batch size to run comfortably on resource-limited notebook setups

train_labels = np.array(train_df["label"])
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = torch.tensor([class_weights[lbl] for lbl in train_labels], dtype=torch.double)

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders created with batch size {BATCH_SIZE}")
print(f"Class representation in train set: {class_counts}")

### Step 7: Load XLM-RoBERTa Model
Load `xlm-roberta-base` with classification heads for 3 output classes.

In [ ]:
print("Loading pre-trained XLM-RoBERTa-base model...")
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)
model.to(device)

### Step 8: Setup Optimizers and warmups

In [ ]:
EPOCHS = 3
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=total_steps
)

# Setup Mixed precision training (conditional scale)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"Autocast training enabled: {use_amp}")

### Step 9: Define Evaluation Metric Helper
A validation function calculates the average loss, accuracy, and macro-average F1 score.

In [ ]:
def run_eval(val_model, dataloader):
    val_model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = val_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, accuracy, macro_f1

### Step 10: Model Training Loop (with Early Stopping)
Trains the model while monitoring validation F1 score. Incorporates early stopping with patience=3.

In [ ]:
best_val_f1 = 0.0
patience = 3
early_stopping_counter = 0
checkpoint_dir = "model/checkpoints/best_model/"
os.makedirs(checkpoint_dir, exist_ok=True)

print("Starting training process...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        with torch.amp.autocast(device_type="cuda" if device.type == "cuda" else "cpu", enabled=use_amp):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        train_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc, val_f1 = run_eval(model, val_loader)
    
    print(f"\n[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        early_stopping_counter = 0
        model.save_pretrained(checkpoint_dir)
        tokenizer.save_pretrained(checkpoint_dir)
        print(f"--> Saved best model checkpoint to '{checkpoint_dir}'")
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= patience:
            print("Early stopping triggered!")
            break

### Step 11: Final Evaluation on Test Set
Loads the best saved checkpoint and calculates classification report, confusion matrix, and One-vs-Rest AUC-ROC curves.

In [ ]:
# Load best checkpoint
best_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
best_model.to(device)
best_model.eval()

all_probs = []
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=-1)
        
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
class_names = ["Negative", "Neutral", "Positive"]

# 1. Classification report
print("\n--- Test Set Metrics ---")
report = classification_report(all_labels, all_preds, target_names=class_names)
print(report)
os.makedirs("assets/", exist_ok=True)
with open("assets/classification_report.txt", "w") as f:
    f.write(report)

# 2. Confusion Matrix Heatmap
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.title("Confusion Matrix Heatmap")
plt.savefig("assets/confusion_matrix.png", dpi=300)
plt.show()

# 3. AUC-ROC Curves
y_bin = label_binarize(all_labels, classes=[0, 1, 2])
plt.figure(figsize=(8, 6))
colors = ["#ff4d4d", "#3399ff", "#2eb82e"]

for i in range(3):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    class_auc = auc(fpr, tpr)
    print(f"{class_names[i]} AUC-ROC: {class_auc:.4f}")
    plt.plot(fpr, tpr, color=colors[i], lw=2, label=f"{class_names[i]} (AUC = {class_auc:.4f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("AUC-ROC Curves (One-vs-Rest)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig("assets/auc_roc_curve.png", dpi=300)
plt.show()

### Step 12: SHAP Explainability & Word Attribution
We implement the subword token aggregation logic to combine SentencePiece tokens back to whole words, and plot the SHAP explanations for three review samples.

In [ ]:
class SentimentExplainer:
    def __init__(self, checkpoint_path: str, device_obj):
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
        self.model.to(device_obj)
        self.model.eval()
        self.device = device_obj
        self.label_map = {0: "negative", 1: "neutral", 2: "positive"}
        
        pipeline_device = 0 if device_obj.type == "cuda" else -1
        self.pred_pipeline = pipeline(
            "text-classification",
            model=self.model,
            tokenizer=self.tokenizer,
            device=pipeline_device,
            top_k=None
        )
        self.explainer = shap.Explainer(self.pred_pipeline)

    def aggregate_subwords(self, tokens: List[str], shap_values: List[float]) -> Dict[str, float]:
        word_scores = {}
        current_word = ""
        current_score = 0.0

        for token, val in zip(tokens, shap_values):
            if token in ["<s>", "</s>", "<pad>", "<unk>"]:
                continue
            if token.startswith(" "):
                if current_word:
                    word_scores[current_word] = word_scores.get(current_word, 0.0) + current_score
                current_word = token.replace(" ", "")
                current_score = val
            else:
                if not current_word:
                    current_word = token
                    current_score = val
                else:
                    current_word += token
                    current_score += val

        if current_word:
            word_scores[current_word] = word_scores.get(current_word, 0.0) + current_score
        return word_scores

    def explain(self, text: str) -> dict:
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1).squeeze(0)
            pred_idx = torch.argmax(probs).item()
            confidence = probs[pred_idx].item()
        
        pred_label = self.label_map[pred_idx]
        shap_values = self.explainer([text])
        explanation = shap_values[0]
        
        class_name = self.model.config.id2label.get(pred_idx, f"LABEL_{pred_idx}")
        if hasattr(explanation, "output_names") and explanation.output_names is not None:
            if class_name in explanation.output_names:
                class_dim = explanation.output_names.index(class_name)
            else:
                class_dim = pred_idx
        else:
            class_dim = pred_idx
            
        raw_scores = explanation.values[:, class_dim].tolist()
        tokens = explanation.data.tolist()
        aggregated_scores = self.aggregate_subwords(tokens, raw_scores)
        
        return {
            "label": pred_label,
            "confidence": float(confidence),
            "shap_scores": aggregated_scores
        }

# Instantiate explainer and test examples
explainer = SentimentExplainer(checkpoint_dir, device)
examples = [
    "This product is amazing! I highly recommend it.",
    "Producto muy malo, una pérdida de dinero.",
    "It is okay, not good not bad."
]

for ex in examples:
    res = explainer.explain(ex)
    print("=" * 60)
    print(f"Text: {ex}")
    print(f"Sentiment: {res['label'].upper()} (Confidence: {res['confidence']:.4f})")
    print("SHAP Word Attribution:")
    for w, val in res["shap_scores"].items():
        print(f"  {w:15s} : {val:+.4f}")